In [ ]:
# @title  { display-mode: "form" }
# @markdown # 1. Test Example Training Clip Generation
# @markdown Since openWakeWord models are trained on synthetic examples of your
# @markdown target wake word, it's a good idea to make sure that the examples
# @markdown sound correct. Type in your target wake word below, and run the
# @markdown cell to listen to it.
# @markdown
# @markdown Here are some tips that can help get the wake word to sound right:

# @markdown - If your wake word isn't being pronounced in the way
# @markdown you want, try spelling out the sounds phonetically with underscores
# @markdown separating each part.
# @markdown For example: "hey siri" --> "hey_seer_e".

# @markdown - Spell out numbers ("2" --> "two")

# @markdown - Avoid all punctuation except for "?" and "!", and remove unicode characters

import os
import sys
from IPython.display import Audio

if not os.path.exists("./piper-sample-generator"):
    !git clone https://github.com/rhasspy/piper-sample-generator
    !wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    !cd piper-sample-generator && git checkout 213d4d5

    # Install system dependencies
    !pip install piper-tts piper-phonemize-cross
    !pip install webrtcvad
    # !pip install 'torch<=2.5' torchvision torchaudio
    !pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu121

target_word = 'Hay Lisa' # @param {type:"string"}

if "piper-sample-generator/" not in sys.path:
    sys.path.append("piper-sample-generator/")
from generate_samples import generate_samples

def text_to_speech(text):
    generate_samples(text = text,
                max_samples=1,
                length_scales=[1.1],
                noise_scales=[0.7], noise_scale_ws = [0.7],
                output_dir = './', batch_size=1, auto_reduce_batch_size=True,
                file_names=["test_generation.wav"]
                )

text_to_speech(target_word)

from IPython.display import Audio
Audio("test_generation.wav", autoplay=True)


In [ ]:
import sys
import numpy as np
import scipy
from scipy import signal, ndimage

# 1. Print environment details
print("Python:", sys.version)
try:
    print("NumPy:", np.__version__)
    print("SciPy:", scipy.__version__)
except Exception as e:
    print("Library version check error:", e)

# 2. Run real functional tests
try:
    x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])

    print("signal test:", signal.convolve(x, [1, 1], mode="valid"))
    print("ndimage test:", ndimage.gaussian_filter(x, sigma=1))

    print("✓ NumPy/SciPy are working correctly")
except Exception as e:
    print("Execution error during tests:", e)


In [ ]:
# ============================================================
# STEP 2 — openWakeWord training environment + data
# ============================================================

import os
import sys
import subprocess
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", os.getcwd())

# ------------------------------------------------------------
# 1. Check NumPy / SciPy
# ------------------------------------------------------------

import numpy as np
import scipy

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)

# ------------------------------------------------------------
# 2. Make sure openWakeWord repository exists
# ------------------------------------------------------------

repo = Path("/content/openwakeword")

if not repo.exists():
    print("\nCloning openWakeWord...")
    !git clone https://github.com/dscripka/openWakeWord.git /content/openwakeword
else:
    print("\n✓ openWakeWord repository already exists")

print("Repository:", repo)

# Check repository structure
if not (repo / "pyproject.toml").exists():
    raise RuntimeError(
        "openWakeWord repository is incomplete. "
        "Delete /content/openwakeword and clone it again."
    )

print("✓ pyproject.toml found")

# ------------------------------------------------------------
# 3. Install openWakeWord WITHOUT changing its dependencies
# ------------------------------------------------------------

print("\nInstalling openWakeWord...")

!pip install -q -e /content/openwakeword --no-deps

print("✓ openWakeWord installed")

# ------------------------------------------------------------
# 4. Install ONLY the training dependencies we need
#
# IMPORTANT:
# We deliberately DO NOT install:
#   numpy
#   scipy
#   onnx2tf
#   ai-edge-litert
#   protobuf
#
# This keeps the working Colab environment intact.
# ------------------------------------------------------------

print("\nInstalling training dependencies...")

!pip install -q \
    mutagen==1.47.0 \
    torchinfo==1.8.0 \
    torchmetrics==1.2.0 \
    speechbrain==0.5.14 \
    audiomentations==0.33.0 \
    torch-audiomentations==0.11.0 \
    acoustics==0.2.6 \
    pronouncing==0.2.0 \
    tqdm

print("✓ Training dependencies installed")

# ------------------------------------------------------------
# 5. Install datasets
# ------------------------------------------------------------

print("\nChecking Hugging Face datasets...")

!pip install -q "datasets>=2.14.6,<4"

print("✓ datasets installed")

# ------------------------------------------------------------
# 6. Install ONNX + onnxscript
#
# These are needed later when exporting the trained model.
# We are NOT installing onnx2tf here.
# ------------------------------------------------------------

print("\nInstalling ONNX tools...")

!pip install -q onnx onnxruntime onnxscript

print("✓ ONNX tools installed")

# ------------------------------------------------------------
# 7. Install speexdsp-ns
# ------------------------------------------------------------

print("\nInstalling speexdsp-ns...")

!pip install -q speexdsp-ns

print("✓ speexdsp-ns installed")

# ------------------------------------------------------------
# 8. Download openWakeWord base models
# ------------------------------------------------------------

print("\nDownloading openWakeWord base models...")

os.chdir("/content")

!wget -q --show-progress \
    https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx

!wget -q --show-progress \
    https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite

!wget -q --show-progress \
    https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx

!wget -q --show-progress \
    https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite

print("✓ Base models downloaded")

# ------------------------------------------------------------
# 9. Create model directory
# ------------------------------------------------------------

Path("/content/models").mkdir(exist_ok=True)

# ------------------------------------------------------------
# 10. Clone Piper sample generator
# ------------------------------------------------------------

piper_dir = Path("/content/piper-sample-generator")

if not piper_dir.exists():
    print("\nCloning Piper sample generator...")
    !git clone -q https://github.com/rhasspy/piper-sample-generator.git /content/piper-sample-generator
else:
    print("\n✓ Piper sample generator already exists")

print("✓ Piper generator ready")

# ------------------------------------------------------------
# 11. Download MIT Room Impulse Response data
# ------------------------------------------------------------

rir_dir = Path("/content/mit_rirs")

if not rir_dir.exists():
    print("\nDownloading MIT RIR dataset...")

    !wget -q --show-progress \
        https://mcdermottlab.mit.edu/Reverb/IRMAudio/Audio/IRs/mit_rirs.tar.bz2 \
        -O /content/mit_rirs.tar.bz2

    print("Extracting MIT RIR dataset...")

    !mkdir -p /content/mit_rirs
    !tar -xjf /content/mit_rirs.tar.bz2 -C /content/mit_rirs

    print("✓ MIT RIR dataset ready")
else:
    print("\n✓ MIT RIR dataset already exists")

# ------------------------------------------------------------
# 12. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL STEP 2 VERIFICATION")
print("=" * 60)

import openwakeword

print("openWakeWord:", openwakeword.__file__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)

try:
    import torch
    print("PyTorch:", torch.__version__)
except Exception as e:
    print("PyTorch ERROR:", e)

try:
    import onnx
    print("ONNX:", onnx.__version__)
except Exception as e:
    print("ONNX ERROR:", e)

try:
    import datasets
    print("datasets:", datasets.__version__)
except Exception as e:
    print("datasets ERROR:", e)

print("\n✓ STEP 2 BASIC ENVIRONMENT SETUP COMPLETE")

In [ ]:
# ============================================
# STEP 3A — Download FMA background audio
# ============================================

import os
import subprocess
from pathlib import Path

fma_dir = Path("/content/fma")
fma_dir.mkdir(parents=True, exist_ok=True)

print("Downloading FMA background audio...")

# Clone a small FMA subset suitable for training backgrounds
if not (fma_dir / ".git").exists():
    subprocess.run([
        "git", "clone",
        "--depth", "1",
        "https://github.com/mdeff/fma.git",
        str(fma_dir)
    ], check=True)

print("\nChecking FMA directory...")

audio_files = list(fma_dir.rglob("*.mp3")) + list(fma_dir.rglob("*.wav"))

print(f"Audio files found: {len(audio_files)}")

if len(audio_files) > 0:
    print("✓ Background audio is available")
    for f in audio_files[:10]:
        print(" ", f)
else:
    print("⚠️ No audio files found yet.")
    print("We will use the proper dataset download method next.")

In [ ]:
# ============================================
# STEP 3B — Download prepared FMA background
# ============================================

import os
import zipfile
import requests
from pathlib import Path

fma_dir = Path("/content/fma")
fma_dir.mkdir(parents=True, exist_ok=True)

url = "https://f002.backblazeb2.com/file/openwakeword-resources/data/fma_sample.zip"
zip_path = "/content/fma_sample.zip"

print("Downloading prepared FMA sample...")
print("This is much smaller than the full FMA dataset.\n")

response = requests.get(url, stream=True)
response.raise_for_status()

total = int(response.headers.get("content-length", 0))
downloaded = 0

with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)
            downloaded += len(chunk)
            if total:
                print(
                    f"\rDownloaded: {downloaded/1024/1024:.1f} / "
                    f"{total/1024/1024:.1f} MB",
                    end=""
                )

print("\n\nExtracting FMA sample...")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content")

# Find audio files anywhere under /content/fma
audio_files = list(fma_dir.rglob("*.mp3")) + list(fma_dir.rglob("*.wav"))

print("\n" + "=" * 50)
print("FMA RESULT")
print("=" * 50)
print("Audio files found:", len(audio_files))

for f in audio_files[:10]:
    print(" ", f)

if len(audio_files) > 0:
    print("\n✓ FMA background audio is ready!")
else:
    print("\n⚠️ No audio files found.")

In [ ]:
# ============================================
# STEP 3C — Locate extracted FMA audio
# ============================================

from pathlib import Path

print("Searching /content for audio files...\n")

audio_extensions = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}

audio_files = []

for p in Path("/content").rglob("*"):
    if p.is_file() and p.suffix.lower() in audio_extensions:
        audio_files.append(p)

print("=" * 60)
print("AUDIO SEARCH RESULT")
print("=" * 60)
print("Total audio files found:", len(audio_files))

for f in audio_files[:30]:
    print(" ", f)

# Show directories containing audio
audio_dirs = sorted(set(str(f.parent) for f in audio_files))

print("\nDirectories containing audio:")
for d in audio_dirs[:20]:
    print(" ", d)

In [ ]:
# ============================================
# STEP 4 — Verify training audio sources
# ============================================

from pathlib import Path
import wave

background_dir = Path("/content/fma_sample")
rir_dir = Path("/content/mit_rirs")

background_files = list(background_dir.glob("*.wav"))
rir_files = list(rir_dir.rglob("*.wav"))

print("=" * 60)
print("TRAINING AUDIO CHECK")
print("=" * 60)

print(f"\nBackground audio:")
print(f"  Directory: {background_dir}")
print(f"  WAV files: {len(background_files)}")

print(f"\nRoom impulse responses:")
print(f"  Directory: {rir_dir}")
print(f"  WAV files: {len(rir_files)}")

# Inspect one background file
if background_files:
    with wave.open(str(background_files[0]), "rb") as w:
        print("\nExample background:")
        print(f"  File: {background_files[0].name}")
        print(f"  Sample rate: {w.getframerate()} Hz")
        print(f"  Channels: {w.getnchannels()}")
        print(f"  Duration: {w.getnframes() / w.getframerate():.2f} sec")

# Inspect one RIR
if rir_files:
    with wave.open(str(rir_files[0]), "rb") as w:
        print("\nExample RIR:")
        print(f"  File: {rir_files[0].name}")
        print(f"  Sample rate: {w.getframerate()} Hz")
        print(f"  Channels: {w.getnchannels()}")
        print(f"  Duration: {w.getnframes() / w.getframerate():.2f} sec")

print("\n" + "=" * 60)

if len(background_files) > 0 and len(rir_files) > 0:
    print("✓ BACKGROUND + RIR DATA READY")
else:
    print("⚠️ Something is missing")

In [ ]:
# ============================================
# STEP 5 — Download validation feature data
# ============================================

from huggingface_hub import hf_hub_download
from pathlib import Path
import os

print("=" * 60)
print("DOWNLOADING VALIDATION FEATURES")
print("=" * 60)

path = hf_hub_download(
    repo_id="davidscripka/openwakeword_features",
    filename="validation_set_features.npy",
    repo_type="dataset",
    local_dir="/content"
)

file_path = Path(path)

print("\n✓ Download complete")
print(f"File: {file_path}")
print(f"Size: {file_path.stat().st_size / (1024**2):.2f} MB")
print(f"Exists: {file_path.exists()}")

print("\n" + "=" * 60)
print("✓ VALIDATION FEATURES READY")
print("=" * 60)

In [ ]:
# ============================================
# STEP 6 — Check Colab storage
# ============================================

import shutil

total, used, free = shutil.disk_usage("/content")

print("=" * 60)
print("COLAB STORAGE CHECK")
print("=" * 60)

print(f"\nTotal space : {total / (1024**3):.2f} GB")
print(f"Used space  : {used / (1024**3):.2f} GB")
print(f"Free space  : {free / (1024**3):.2f} GB")

print("\n" + "=" * 60)

if free / (1024**3) >= 25:
    print("✓ Enough free space to consider the ACAV dataset")
else:
    print("⚠️ Free space is low — we should use a different approach")

In [ ]:
# ============================================
# STEP 7 — Download ACAV training features
# ============================================

from huggingface_hub import hf_hub_download
from pathlib import Path

print("=" * 60)
print("DOWNLOADING ACAV TRAINING FEATURES")
print("=" * 60)

print("\nThis is a large file (~17 GB).")
print("Please let the download finish completely.\n")

path = hf_hub_download(
    repo_id="davidscripka/openwakeword_features",
    filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
    repo_type="dataset",
    local_dir="/content"
)

file_path = Path(path)

print("\n" + "=" * 60)
print("✓ ACAV DOWNLOAD COMPLETE")
print("=" * 60)

print(f"\nFile: {file_path}")
print(f"Size: {file_path.stat().st_size / (1024**3):.2f} GB")
print(f"Exists: {file_path.exists()}")

In [ ]:
# ============================================
# STEP 8 — Final training data verification
# ============================================

from pathlib import Path

print("=" * 60)
print("FINAL TRAINING DATA CHECK")
print("=" * 60)

checks = {
    "Background audio": Path("/content/fma_sample"),
    "MIT RIR": Path("/content/mit_rirs"),
    "Validation features": Path("/content/validation_set_features.npy"),
    "ACAV features": Path("/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"),
    "Piper generator": Path("/content/piper-sample-generator/generate_samples.py"),
    "openWakeWord": Path("/content/openwakeword"),
}

all_ready = True

for name, path in checks.items():
    exists = path.exists()

    if exists:
        print(f"✓ {name}: READY")
    else:
        print(f"✗ {name}: MISSING")
        all_ready = False

print("\n" + "=" * 60)

if all_ready:
    print("✓ ALL REQUIRED TRAINING COMPONENTS ARE READY")
else:
    print("⚠️ SOMETHING IS MISSING")

print("=" * 60)

In [ ]:
# ============================================
# STEP 9 — Create Hey Lisa training config
# ============================================

import yaml
from pathlib import Path

config_path = Path("/content/openwakeword/examples/custom_model.yml")

print("=" * 60)
print("CREATING HEY LISA TRAINING CONFIG")
print("=" * 60)

# Load the official openWakeWord training configuration
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# --------------------------------------------
# Our custom settings
# --------------------------------------------

config["target_phrase"] = ["Hey Lisa"]

# Model filename will become:
# hey_lisa.onnx
config["model_name"] = "hey_lisa"

# First training run:
# Keep this moderate so we can verify the pipeline.
config["n_samples"] = 1000
config["n_samples_val"] = 500
config["steps"] = 10000

# Detection targets
config["target_accuracy"] = 0.5
config["target_recall"] = 0.25

# Output
config["output_dir"] = "/content/my_custom_model"

# False-positive penalty
config["max_negative_weight"] = 1500

# --------------------------------------------
# Actual datasets we downloaded
# --------------------------------------------

config["background_paths"] = [
    "/content/fma_sample"
]

config["rir_paths"] = [
    "/content/mit_rirs"
]

config["false_positive_validation_data_path"] = (
    "/content/validation_set_features.npy"
)

config["feature_data_files"] = {
    "ACAV100M_sample":
        "/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
}

# --------------------------------------------
# Save configuration
# --------------------------------------------

output_config = "/content/hey_lisa_model.yaml"

with open(output_config, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("\n✓ Configuration created")
print(f"Config file: {output_config}")

print("\nKey settings:")
print(f"  Target phrase : {config['target_phrase']}")
print(f"  Model name    : {config['model_name']}")
print(f"  Samples       : {config['n_samples']}")
print(f"  Validation    : {config['n_samples_val']}")
print(f"  Steps         : {config['steps']}")
print(f"  Background    : {config['background_paths']}")
print(f"  RIR           : {config['rir_paths']}")

print("\n" + "=" * 60)
print("✓ HEY LISA CONFIG READY")
print("=" * 60)

In [ ]:
# ============================================
# STEP 10 — Validate Hey Lisa configuration
# ============================================

import yaml
from pathlib import Path

config_file = Path("/content/hey_lisa_model.yaml")

print("=" * 60)
print("VALIDATING HEY LISA CONFIGURATION")
print("=" * 60)

with open(config_file, "r") as f:
    config = yaml.safe_load(f)

# Required paths
required_paths = {
    "Background": config["background_paths"],
    "RIR": config["rir_paths"],
    "Validation": [config["false_positive_validation_data_path"]],
    "ACAV": list(config["feature_data_files"].values()),
}

all_ok = True

print("\nChecking paths...")

for category, paths in required_paths.items():
    for path_str in paths:
        path = Path(path_str)

        if path.exists():
            print(f"✓ {category}: {path}")
        else:
            print(f"✗ {category}: MISSING — {path}")
            all_ok = False

print("\nChecking target phrase...")

if config["target_phrase"] == ["Hey Lisa"]:
    print("✓ Target phrase: Hey Lisa")
else:
    print(f"✗ Unexpected target phrase: {config['target_phrase']}")
    all_ok = False

print("\nChecking model name...")

if config["model_name"] == "hey_lisa":
    print("✓ Model name: hey_lisa")
else:
    print(f"✗ Unexpected model name: {config['model_name']}")
    all_ok = False

print("\n" + "=" * 60)

if all_ok:
    print("✓ CONFIGURATION VALIDATION PASSED")
    print("\nReady for the next training preparation step.")
else:
    print("⚠️ CONFIGURATION HAS PROBLEMS")

print("=" * 60)

In [ ]:
# ============================================
# STEP 11 — Generate Hey Lisa training clips
# ============================================

import sys
import subprocess
from pathlib import Path

config_file = "/content/hey_lisa_model.yaml"

print("=" * 60)
print("GENERATING HEY LISA TRAINING CLIPS")
print("=" * 60)

print("\nTarget: Hey Lisa")
print("Training positives: 1000")
print("Validation positives: 500")
print("Adversarial negatives: generated automatically")
print("\nPlease wait for the generation process to finish...\n")

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        config_file,
        "--generate_clips",
    ],
    text=True
)

print("\n" + "=" * 60)

if result.returncode == 0:
    print("✓ CLIP GENERATION COMPLETED")
else:
    print(f"⚠️ CLIP GENERATION EXITED WITH CODE {result.returncode}")

print("=" * 60)

output_dir = Path("/content/my_custom_model/hey_lisa")

if output_dir.exists():
    print("\nGenerated directories:")

    for directory in sorted(output_dir.iterdir()):
        if directory.is_dir():
            wav_count = len(list(directory.glob("*.wav")))
            print(f"  {directory.name}: {wav_count} WAV files")

In [ ]:
# ============================================
# STEP 12 — Augment Hey Lisa training clips
# ============================================

import sys
import subprocess

config_file = "/content/hey_lisa_model.yaml"

print("=" * 60)
print("AUGMENTING HEY LISA TRAINING CLIPS")
print("=" * 60)

print("\nUsing:")
print("  ✓ FMA background audio")
print("  ✓ MIT room impulse responses")
print("  ✓ Generated Hey Lisa clips")

print("\nThis may take some time.")
print("Please let it finish completely.\n")

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        config_file,
        "--augment_clips",
    ],
    text=True
)

print("\n" + "=" * 60)

if result.returncode == 0:
    print("✓ AUDIO AUGMENTATION COMPLETED")
else:
    print(f"⚠️ AUGMENTATION EXITED WITH CODE {result.returncode}")

print("=" * 60)

In [ ]:
# ============================================
# STEP 12A — Diagnose augmentation error
# ============================================

import sys
import subprocess

config_file = "/content/hey_lisa_model.yaml"

print("=" * 60)
print("DIAGNOSING AUGMENTATION")
print("=" * 60)
print()

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        config_file,
        "--augment_clips",
    ]
)

print("\n" + "=" * 60)
print(f"PROCESS EXIT CODE: {result.returncode}")
print("=" * 60)

In [ ]:
from pathlib import Path

print("=" * 60)
print("CHECKING MIT RIR FOLDER")
print("=" * 60)

root = Path("/content/mit_rirs")

print("\nTop-level contents:")
for item in root.iterdir():
    print(" ", item, " [DIR]" if item.is_dir() else " [FILE]")

print("\nRIR WAV files:")
wav_files = list(root.rglob("*.wav"))

print("Total WAV files:", len(wav_files))

print("\nFirst 10 WAV files:")
for f in wav_files[:10]:
    print(" ", f)

print("\n" + "=" * 60)

In [ ]:
import yaml

config_file = "/content/hey_lisa_model.yaml"

# Load current configuration
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

# Fix RIR path
config["rir_paths"] = ["/content/mit_rirs/MIT_Survey"]

# Save corrected configuration
with open(config_file, "w") as f:
    yaml.dump(config, f, sort_keys=False)

print("=" * 60)
print("RIR PATH FIXED")
print("=" * 60)
print()
print("New RIR path:")
print(config["rir_paths"][0])
print()
print("✓ Configuration saved")

In [ ]:
import sys
import subprocess

config_file = "/content/hey_lisa_model.yaml"

print("=" * 60)
print("STARTING HEY LISA AUGMENTATION")
print("=" * 60)
print()
print("Using RIR path:")
print("/content/mit_rirs/MIT_Survey")
print()
print("Please wait...")
print()

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        config_file,
        "--augment_clips",
    ],
    text=True
)

print()
print("=" * 60)
print("AUGMENTATION FINISHED")
print("=" * 60)
print()
print("Exit code:", result.returncode)

if result.returncode == 0:
    print("✓ AUGMENTATION SUCCESSFUL")
else:
    print("❌ AUGMENTATION FAILED")
    print("The error above is important — send me the full output.")

In [ ]:
from pathlib import Path

train_file = Path("/content/openwakeword/openwakeword/train.py")

print("=" * 60)
print("CHECKING AUGMENTATION CODE")
print("=" * 60)

text = train_file.read_text()

# Find the augmentation section
keywords = [
    "augment_clips",
    "background_clip_paths",
    "RIR_paths",
    "augmentation_rounds"
]

for keyword in keywords:
    print(f"\n--- Searching for: {keyword} ---")
    positions = [i for i in range(len(text)) if text.startswith(keyword, i)]
    print("Occurrences:", len(positions))
    for pos in positions[:3]:
        start = max(0, pos - 500)
        end = min(len(text), pos + 1000)
        print(text[start:end])
        print("\n" + "-" * 40)

print("\n" + "=" * 60)
print("INSPECTION COMPLETE")
print("=" * 60)

In [ ]:
import sys
import traceback
from pathlib import Path

# Make sure openWakeWord is available
sys.path.insert(0, "/content/openwakeword")

from openwakeword.data import augment_clips

print("=" * 60)
print("DIRECT AUGMENTATION TEST")
print("=" * 60)

# Find one generated positive clip
clips = list(Path("/content/my_custom_model/positive_train").glob("*.wav"))

print("Generated clips found:", len(clips))

if not clips:
    print("❌ No positive training clips found.")
else:
    test_clip = str(clips[0])

    print("Test clip:")
    print(test_clip)

    print("\nTesting:")
    print("Background:", "/content/fma_sample")
    print("RIR:", "/content/mit_rirs/MIT_Survey")
    print()

    try:
        generator = augment_clips(
            [test_clip],
            total_length=32000,
            batch_size=1,
            background_clip_paths=[
                str(p) for p in Path("/content/fma_sample").glob("*.wav")
            ],
            RIR_paths=[
                str(p) for p in Path("/content/mit_rirs/MIT_Survey").glob("*.wav")
            ]
        )

        print("Generator created successfully.")
        print("Requesting first augmented sample...")

        sample = next(generator)

        print()
        print("✓ DIRECT AUGMENTATION TEST PASSED")
        print("Output type:", type(sample))

    except Exception as e:
        print()
        print("=" * 60)
        print("❌ DIRECT AUGMENTATION ERROR")
        print("=" * 60)
        traceback.print_exc()

ModuleNotFoundError: No module named 'openwakeword.data'

In [ ]:
from pathlib import Path

root = Path("/content/openwakeword")

print("=" * 60)
print("SEARCHING FOR AUGMENT_CLIPS")
print("=" * 60)
print()

matches = []

for file in root.rglob("*.py"):
    try:
        text = file.read_text(errors="ignore")
        if "def augment_clips" in text or "augment_clips =" in text:
            matches.append(file)
    except:
        pass

print("Files containing augment_clips:")
print()

for file in matches:
    print("✓", file)

print()
print("=" * 60)
print("SEARCHING FOR DATA MODULE")
print("=" * 60)
print()

data_items = list((root / "openwakeword").glob("*"))

for item in data_items:
    print(item)

print()
print("=" * 60)

SEARCHING FOR AUGMENT_CLIPS

Files containing augment_clips:

✓ /content/openwakeword/openwakeword/data.py

SEARCHING FOR DATA MODULE

/content/openwakeword/openwakeword/__pycache__
/content/openwakeword/openwakeword/model.py
/content/openwakeword/openwakeword/custom_verifier_model.py
/content/openwakeword/openwakeword/__init__.py
/content/openwakeword/openwakeword/utils.py
/content/openwakeword/openwakeword/data.py
/content/openwakeword/openwakeword/metrics.py
/content/openwakeword/openwakeword/vad.py
/content/openwakeword/openwakeword/train.py



In [ ]:
import sys
import importlib.util

print("=" * 60)
print("LOADING OPENWAKEWORD DATA MODULE")
print("=" * 60)
print()

data_file = "/content/openwakeword/openwakeword/data.py"

spec = importlib.util.spec_from_file_location(
    "openwakeword_data",
    data_file
)

data_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_module)

print("✓ data.py loaded successfully")
print()

print("augment_clips available:",
      hasattr(data_module, "augment_clips"))

print("mmap_batch_generator available:",
      hasattr(data_module, "mmap_batch_generator"))

print()
print("=" * 60)

LOADING OPENWAKEWORD DATA MODULE



/usr/local/lib/python3.12/dist-packages/torch_audiomentations/utils/io.py:27: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


✓ data.py loaded successfully

augment_clips available: True
mmap_batch_generator available: True



In [ ]:
from pathlib import Path
import traceback

print("=" * 60)
print("TESTING ONE HEY LISA AUGMENTATION")
print("=" * 60)
print()

# Find generated Hey Lisa clip
clips = list(
    Path("/content/my_custom_model/positive_train").glob("*.wav")
)

print("Positive clips found:", len(clips))

if len(clips) == 0:
    print("❌ No positive clips found.")
else:
    test_clip = str(clips[0])

    print("Test clip:")
    print(test_clip)
    print()

    # Get ONLY WAV files
    background_files = [
        str(p)
        for p in Path("/content/fma_sample").glob("*.wav")
    ]

    rir_files = [
        str(p)
        for p in Path("/content/mit_rirs/MIT_Survey").glob("*.wav")
    ]

    print("Background WAV files:", len(background_files))
    print("RIR WAV files:", len(rir_files))
    print()

    try:
        generator = data_module.augment_clips(
            [test_clip],
            total_length=32000,
            batch_size=1,
            background_clip_paths=background_files,
            RIR_paths=rir_files
        )

        print("✓ Augmentation generator created")
        print("Generating one augmented sample...")
        print()

        sample = next(generator)

        print("=" * 60)
        print("✓ AUGMENTATION TEST PASSED")
        print("=" * 60)
        print()
        print("Output type:", type(sample))

    except Exception:
        print("=" * 60)
        print("❌ AUGMENTATION ERROR")
        print("=" * 60)
        print()
        traceback.print_exc()

In [ ]:
from pathlib import Path

print("=" * 60)
print("FINDING GENERATED HEY LISA CLIPS")
print("=" * 60)
print()

root = Path("/content/my_custom_model")

print("Does my_custom_model exist?", root.exists())
print()

if root.exists():
    print("Folders/files inside my_custom_model:")
    for item in root.iterdir():
        print(" ", item)

print()
print("Searching for WAV files...")
print()

wav_files = list(root.rglob("*.wav"))

print("Total WAV files found:", len(wav_files))

print()
print("First 20 WAV files:")

for f in wav_files[:20]:
    print(" ", f)

print()
print("=" * 60)


In [ ]:
from pathlib import Path
import traceback

print("=" * 60)
print("TESTING ONE HEY LISA AUGMENTATION")
print("=" * 60)
print()

# Correct location
clips = list(
    Path("/content/my_custom_model/hey_lisa/positive_train").glob("*.wav")
)

print("Positive training clips found:", len(clips))

if not clips:
    print("❌ No positive training clips found.")
else:
    test_clip = str(clips[0])

    print("Test clip:")
    print(test_clip)
    print()

    background_files = [
        str(p)
        for p in Path("/content/fma_sample").glob("*.wav")
    ]

    rir_files = [
        str(p)
        for p in Path("/content/mit_rirs/MIT_Survey").glob("*.wav")
    ]

    print("Background WAV files:", len(background_files))
    print("RIR WAV files:", len(rir_files))
    print()

    try:
        generator = data_module.augment_clips(
            [test_clip],
            total_length=32000,
            batch_size=1,
            background_clip_paths=background_files,
            RIR_paths=rir_files
        )

        print("✓ Augmentation generator created")
        print("Generating one augmented sample...")
        print()

        sample = next(generator)

        print("=" * 60)
        print("✓ AUGMENTATION TEST PASSED")
        print("=" * 60)
        print()
        print("Output type:", type(sample))

    except Exception:
        print("=" * 60)
        print("❌ AUGMENTATION ERROR")
        print("=" * 60)
        print()
        traceback.print_exc()

In [ ]:
import sys
import subprocess

print("=" * 60)
print("STEP 13 — FULL HEY LISA AUGMENTATION")
print("=" * 60)
print()
print("Starting full augmentation...")
print()
print("Using:")
print("  ✓ Hey Lisa training clips")
print("  ✓ FMA background audio")
print("  ✓ MIT RIR")
print()
print("This may take some time.")
print("Do NOT restart the Colab runtime while it is running.")
print()

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        "/content/hey_lisa_model.yaml",
        "--augment_clips"
    ],
    text=True
)

print()
print("=" * 60)

if result.returncode == 0:
    print("✓ STEP 13 AUGMENTATION COMPLETED")
else:
    print("❌ STEP 13 FAILED")
    print("Exit code:", result.returncode)

print("=" * 60)

In [ ]:
import sys
import subprocess

print("=" * 60)
print("STEP 13A — DIAGNOSE FULL AUGMENTATION")
print("=" * 60)
print()

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        "/content/hey_lisa_model.yaml",
        "--augment_clips"
    ],
    text=True
)

print()
print("=" * 60)
print("PROCESS FINISHED")
print("=" * 60)
print("Exit code:", result.returncode)

In [ ]:
import sys
import openwakeword

print("=" * 60)
print("STEP 13B — CHECK OPENWAKEWORD IMPORT")
print("=" * 60)
print()

print("Python executable:")
print(sys.executable)
print()

print("openwakeword location:")
print(openwakeword.__file__)
print()

print("openwakeword package paths:")
print(list(openwakeword.__path__))
print()

print("Python path:")
for p in sys.path[:10]:
    print(" ", p)

print()
print("=" * 60)

In [ ]:
import sys
import os

OPENWAKEWORD_ROOT = "/content/openwakeword"

# Put the local repository first
if OPENWAKEWORD_ROOT in sys.path:
    sys.path.remove(OPENWAKEWORD_ROOT)

sys.path.insert(0, OPENWAKEWORD_ROOT)

# Remove any previously loaded openwakeword modules
for name in list(sys.modules.keys()):
    if name == "openwakeword" or name.startswith("openwakeword."):
        del sys.modules[name]

print("=" * 60)
print("STEP 13C — FIXING OPENWAKEWORD IMPORT")
print("=" * 60)
print()

import openwakeword

print("openwakeword loaded")
print("Location:", openwakeword.__file__)
print("Paths:", list(openwakeword.__path__))
print()

try:
    from openwakeword.data import augment_clips

    print("✓ openwakeword.data imported successfully")
    print("✓ augment_clips imported successfully")

except Exception as e:
    print("❌ Import still failed")
    print(type(e).__name__ + ":", e)

print()
print("=" * 60)

In [ ]:
import sys
import subprocess
import os

print("=" * 60)
print("STEP 13D — FULL HEY LISA AUGMENTATION")
print("=" * 60)
print()

# Make sure the local openWakeWord source is used
env = os.environ.copy()
env["PYTHONPATH"] = "/content/openwakeword:/content"

print("Using local openWakeWord source:")
print("  /content/openwakeword")
print()

print("Starting augmentation...")
print("Please wait — do not restart Colab.")
print()

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        "/content/hey_lisa_model.yaml",
        "--augment_clips"
    ],
    env=env,
    text=True
)

print()
print("=" * 60)

if result.returncode == 0:
    print("✓ STEP 13 COMPLETED SUCCESSFULLY")
else:
    print("❌ STEP 13 FAILED")
    print("Exit code:", result.returncode)

print("=" * 60)

In [ ]:
from pathlib import Path
import traceback
import yaml

print("=" * 60)
print("STEP 13E — DIRECT FULL AUGMENTATION DIAGNOSTIC")
print("=" * 60)
print()

try:
    # Load configuration
    with open("/content/hey_lisa_model.yaml", "r") as f:
        config = yaml.safe_load(f)

    print("✓ Configuration loaded")
    print()

    # Load the already-tested local data module
    import openwakeword.data as data

    print("✓ openwakeword.data loaded")
    print()

    # Paths
    base = Path("/content/my_custom_model/hey_lisa")

    background_files = [
        str(p) for p in Path("/content/fma_sample").glob("*.wav")
    ]

    rir_files = [
        str(p) for p in Path("/content/mit_rirs/MIT_Survey").glob("*.wav")
    ]

    print("Background files:", len(background_files))
    print("RIR files:", len(rir_files))
    print()

    folders = [
        "positive_train",
        "positive_test",
        "negative_train",
        "negative_test"
    ]

    for folder in folders:

        clips = [
            str(p) for p in (base / folder).glob("*.wav")
        ]

        print("-" * 60)
        print("Testing folder:", folder)
        print("Clips:", len(clips))

        if not clips:
            print("❌ No clips found")
            continue

        # Test a small batch first
        test_clips = clips[:10]

        print("Testing first", len(test_clips), "clips...")

        generator = data.augment_clips(
            test_clips,
            total_length=32000,
            batch_size=1,
            background_clip_paths=background_files,
            RIR_paths=rir_files
        )

        for i in range(len(test_clips)):
            sample = next(generator)

            if i == 0:
                print("  First output:", type(sample))

        print("✓", folder, "augmentation test passed")

    print()
    print("=" * 60)
    print("✓ DIRECT AUGMENTATION DIAGNOSTIC PASSED")
    print("=" * 60)

except Exception:
    print()
    print("=" * 60)
    print("❌ DIRECT AUGMENTATION ERROR")
    print("=" * 60)
    print()
    traceback.print_exc()

In [ ]:
from pathlib import Path
import wave

print("=" * 60)
print("STEP 13F — CHECK HEY LISA WAV SAMPLE RATES")
print("=" * 60)
print()

folders = [
    "positive_train",
    "positive_test",
    "negative_train",
    "negative_test"
]

base = Path("/content/my_custom_model/hey_lisa")

for folder in folders:
    files = list((base / folder).glob("*.wav"))

    print("-" * 60)
    print(folder)
    print("Files:", len(files))

    if files:
        for wav_file in files[:3]:
            try:
                with wave.open(str(wav_file), "rb") as wf:
                    sample_rate = wf.getframerate()
                    channels = wf.getnchannels()
                    sample_width = wf.getsampwidth()
                    frames = wf.getnframes()

                duration = frames / sample_rate

                print()
                print("File:", wav_file.name)
                print("  Sample rate:", sample_rate, "Hz")
                print("  Channels:", channels)
                print("  Sample width:", sample_width, "bytes")
                print("  Duration:", round(duration, 3), "seconds")

            except Exception as e:
                print("❌ Could not read:", wav_file)
                print(type(e).__name__ + ":", e)

print()
print("=" * 60)
print("CHECK COMPLETE")
print("=" * 60)

In [ ]:
from pathlib import Path
import wave

print("=" * 60)
print("STEP 13G — CHECK BACKGROUND AND RIR SAMPLE RATES")
print("=" * 60)
print()

def check_files(label, files):
    print("-" * 60)
    print(label)
    print("Files:", len(files))
    print()

    sample_rates = {}
    channels = {}

    for wav_file in files[:20]:
        try:
            with wave.open(str(wav_file), "rb") as wf:
                sr = wf.getframerate()
                ch = wf.getnchannels()

            sample_rates[sr] = sample_rates.get(sr, 0) + 1
            channels[ch] = channels.get(ch, 0) + 1

        except Exception as e:
            print("❌ Error:", wav_file.name, e)

    print("Sample rates found:", sample_rates)
    print("Channels found:", channels)
    print()

    if sample_rates:
        for sr in sample_rates:
            print("  ", sr, "Hz")

    if channels:
        for ch in channels:
            print("  ", ch, "channel(s)")


# FMA background audio
fma_files = list(
    Path("/content/fma_sample").glob("*.wav")
)

# MIT RIR audio
rir_files = list(
    Path("/content/mit_rirs/MIT_Survey").glob("*.wav")
)

check_files("FMA BACKGROUND AUDIO", fma_files)
check_files("MIT RIR AUDIO", rir_files)

print("=" * 60)
print("CHECK COMPLETE")
print("=" * 60)

In [ ]:
from pathlib import Path
import wave
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly

print("=" * 60)
print("STEP 13H — CONVERT MIT RIR TO 16 kHz")
print("=" * 60)
print()

source_dir = Path("/content/mit_rirs/MIT_Survey")
output_dir = Path("/content/mit_rirs_16k")
output_dir.mkdir(parents=True, exist_ok=True)

rir_files = list(source_dir.glob("*.wav"))

print("Original RIR files:", len(rir_files))
print("Output folder:", output_dir)
print()

converted = 0
skipped = 0

for i, src in enumerate(rir_files, 1):

    dst = output_dir / src.name

    if dst.exists():
        skipped += 1
        continue

    try:
        sample_rate, audio = wavfile.read(str(src))

        # Convert stereo to mono if necessary
        if audio.ndim > 1:
            audio = audio.mean(axis=1)

        # Convert to float for resampling
        audio = audio.astype(np.float32)

        # 32,000 Hz -> 16,000 Hz
        if sample_rate != 16000:
            audio = resample_poly(audio, 16000, sample_rate)

        # Prevent clipping
        audio = np.clip(audio, -32768, 32767)

        # Save as 16-bit PCM WAV
        audio_int16 = audio.astype(np.int16)

        wavfile.write(
            str(dst),
            16000,
            audio_int16
        )

        converted += 1

    except Exception as e:
        print()
        print("❌ Error converting:", src.name)
        print(type(e).__name__ + ":", e)

    if i % 25 == 0:
        print(f"Processed {i}/{len(rir_files)}")

print()
print("=" * 60)
print("CONVERSION COMPLETE")
print("=" * 60)
print()
print("Converted:", converted)
print("Already existed:", skipped)
print("Output files:", len(list(output_dir.glob("*.wav"))))
print()

In [ ]:
import yaml

print("=" * 60)
print("STEP 13I — UPDATE RIR PATH")
print("=" * 60)
print()

config_path = "/content/hey_lisa_model.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config["rir_paths"] = ["/content/mit_rirs_16k"]

with open(config_path, "w") as f:
    yaml.dump(config, f, sort_keys=False)

print("✓ Configuration updated")
print()
print("New RIR path:")
print(" ", config["rir_paths"][0])
print()

print("=" * 60)
print("✓ STEP 13I COMPLETE")
print("=" * 60)

In [ ]:
from pathlib import Path
from scipy.io import wavfile

print("=" * 60)
print("STEP 13J — VERIFY 16 kHz RIR FILES")
print("=" * 60)
print()

rir_dir = Path("/content/mit_rirs_16k")
rir_files = list(rir_dir.glob("*.wav"))

sample_rates = {}

for wav in rir_files:
    sr, _ = wavfile.read(str(wav))
    sample_rates[sr] = sample_rates.get(sr, 0) + 1

print("RIR files found:", len(rir_files))
print("Sample rates:", sample_rates)
print()

if len(rir_files) == 270 and sample_rates == {16000: 270}:
    print("✓ ALL 270 RIR FILES ARE 16 kHz")
    print("✓ RIR DATA IS READY FOR AUGMENTATION")
else:
    print("❌ RIR verification failed")
    print("Please send me this output before continuing.")

print()
print("=" * 60)
print("✓ STEP 13J COMPLETE")
print("=" * 60)


In [ ]:
import sys
import yaml
from pathlib import Path
import importlib.util
import wave

print("=" * 60)
print("STEP 13K — FINAL AUGMENTATION TEST")
print("=" * 60)
print()

# Make sure local openWakeWord is used
if "/content/openwakeword" not in sys.path:
    sys.path.insert(0, "/content/openwakeword")

# Load openWakeWord data.py directly
data_path = "/content/openwakeword/openwakeword/data.py"

spec = importlib.util.spec_from_file_location("oww_data", data_path)
data_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(data_module)

print("✓ openWakeWord data module loaded")

# Load configuration
with open("/content/hey_lisa_model.yaml", "r") as f:
    config = yaml.safe_load(f)

background_paths = config["background_paths"]
rir_paths = config["rir_paths"]

print("Background:", background_paths)
print("RIR:", rir_paths)
print()

# Find positive training clips
positive_dir = Path("/content/my_custom_model/hey_lisa/positive_train")
positive_clips = list(positive_dir.glob("*.wav"))

print("Positive training clips:", len(positive_clips))

# Find positive test clip
positive_test_dir = Path("/content/my_custom_model/hey_lisa/positive_test")
positive_test_clips = list(positive_test_dir.glob("*.wav"))

if not positive_test_clips:
    raise RuntimeError("No positive test clips found.")

test_clip = positive_test_clips[0]

with wave.open(str(test_clip), "rb") as wf:
    sample_rate = wf.getframerate()
    num_frames = wf.getnframes()

total_length = max(num_frames, 32000)

print("Sample rate:", sample_rate)
print("Test clip frames:", num_frames)
print("Augmentation total_length:", total_length)
print()

# Use 10 clips
test_clips = [str(x) for x in positive_clips[:10]]

print("Input clips:", len(test_clips))
print()

try:
    generator = data_module.augment_clips(
        test_clips,
        total_length=total_length,
        batch_size=config["augmentation_batch_size"],
        background_clip_paths=background_paths,
        RIR_paths=rir_paths
    )

    print("✓ Augmentation generator created")
    print()
    print("Generating ONE batch...")

    batch = next(generator)

    print()
    print("✓ Batch generated successfully")
    print("Batch type:", type(batch))
    print("Batch shape:", batch.shape)
    print("Batch dtype:", batch.dtype)
    print()

    expected_shape = (
        config["augmentation_batch_size"],
        total_length
    )

    print("Expected batch size:", config["augmentation_batch_size"])
    print("Expected length:", total_length)

    print()
    print("=" * 60)

    if batch.shape[1] == total_length:
        print("🎉 STEP 13K PASSED")
        print("=" * 60)
        print()
        print("✓ RIR sample-rate problem is FIXED")
        print("✓ Background audio works")
        print("✓ RIR audio works")
        print("✓ Augmentation works")
        print("✓ Batch shape is correct")
        print()
        print("We are ready for FULL AUGMENTATION.")
    else:
        print("❌ Unexpected batch shape")
        print("=" * 60)

except Exception as e:
    print()
    print("=" * 60)
    print("❌ STEP 13K FAILED")
    print("=" * 60)
    print()
    print(type(e).__name__ + ":", e)
    raise

In [ ]:
import sys
import os

print("=" * 60)
print("STEP 13L — FULL AUGMENTATION")
print("=" * 60)
print()
print("Starting full augmentation...")
print("This may take some time.")
print()

# Ensure the local openWakeWord package is found first
os.environ["PYTHONPATH"] = "/content/openwakeword:/content"

# Run augmentation
result = os.system(
    f"{sys.executable} "
    "/content/openwakeword/openwakeword/train.py "
    "--training_config /content/hey_lisa_model.yaml "
    "--augment_clips"
)

print()
print("=" * 60)

if result == 0:
    print("🎉 STEP 13L PASSED")
    print("=" * 60)
    print()
    print("✓ Full augmentation completed")
    print("✓ Training audio is ready")
else:
    print("❌ STEP 13L FAILED")
    print("=" * 60)
    print()
    print("Exit code:", result)
    print()
    print("Send me the complete output.")

In [ ]:
import sys
import os
import subprocess

print("=" * 60)
print("STEP 13M — FULL AUGMENTATION ERROR DIAGNOSTIC")
print("=" * 60)
print()

env = os.environ.copy()
env["PYTHONPATH"] = "/content/openwakeword:/content"

cmd = [
    sys.executable,
    "-u",
    "/content/openwakeword/openwakeword/train.py",
    "--training_config",
    "/content/hey_lisa_model.yaml",
    "--augment_clips"
]

print("Running:")
print(" ".join(cmd))
print()
print("-" * 60)
print("TRAINING OUTPUT")
print("-" * 60)
print()

process = subprocess.run(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print(process.stdout)

print()
print("-" * 60)
print("END TRAINING OUTPUT")
print("-" * 60)
print()
print("Process return code:", process.returncode)
print()

if process.returncode == 0:
    print("🎉 FULL AUGMENTATION PASSED")
else:
    print("❌ FULL AUGMENTATION FAILED")
    print()
    print("The complete error above is what we need to fix.")

In [ ]:
from pathlib import Path

print("=" * 60)
print("STEP 13N — LOCATE OPENWAKEWORD MODEL FILES")
print("=" * 60)
print()

models = [
    "melspectrogram.onnx",
    "embedding_model.onnx"
]

search_roots = [
    Path("/content"),
    Path("/usr/local/lib/python3.12/dist-packages/openwakeword")
]

for model in models:
    print(f"Searching for: {model}")

    found = []

    for root in search_roots:
        if root.exists():
            found.extend(root.rglob(model))

    # Remove duplicates
    found = list(dict.fromkeys(found))

    if found:
        for path in found:
            print("  ✓", path)
    else:
        print("  ❌ Not found")

    print()

print("=" * 60)
print("STEP 13N COMPLETE")
print("=" * 60)

In [ ]:
from pathlib import Path
import shutil

print("=" * 60)
print("STEP 13O — INSTALL OPENWAKEWORD MODEL FILES")
print("=" * 60)
print()

source_dir = Path("/content")
target_dir = Path("/content/openwakeword/openwakeword/resources/models")

target_dir.mkdir(parents=True, exist_ok=True)

models = [
    "melspectrogram.onnx",
    "embedding_model.onnx"
]

for model in models:
    source = source_dir / model
    target = target_dir / model

    if not source.exists():
        print(f"❌ Missing source: {source}")
        continue

    shutil.copy2(source, target)

    print(f"✓ Copied: {model}")
    print(f"  → {target}")

print()
print("=" * 60)
print("VERIFYING")
print("=" * 60)
print()

all_ok = True

for model in models:
    path = target_dir / model

    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"✓ {model}")
        print(f"  Size: {size_mb:.2f} MB")
    else:
        print(f"❌ {model} NOT FOUND")
        all_ok = False

print()

if all_ok:
    print("🎉 STEP 13O PASSED")
    print("✓ Both ONNX models are in the location train.py expects")
else:
    print("❌ STEP 13O FAILED")

print("=" * 60)

In [ ]:
import onnxruntime as ort
from pathlib import Path

print("=" * 60)
print("STEP 13P — VERIFY ONNX MODELS")
print("=" * 60)
print()

models_dir = Path(
    "/content/openwakewakeword/openwakeword/resources/models"
)

# Correct the path
models_dir = Path(
    "/content/openwakeword/openwakeword/resources/models"
)

models = [
    "melspectrogram.onnx",
    "embedding_model.onnx"
]

all_ok = True

for model in models:
    path = models_dir / model

    print("Testing:", model)

    try:
        session = ort.InferenceSession(
            str(path),
            providers=["CPUExecutionProvider"]
        )

        print("  ✓ File exists")
        print("  ✓ ONNX Runtime loaded successfully")
        print("  Input:", session.get_inputs()[0].name)
        print("  Output:", session.get_outputs()[0].name)

    except Exception as e:
        print("  ❌ FAILED")
        print(" ", type(e).__name__ + ":", e)
        all_ok = False

    print()

print("=" * 60)

if all_ok:
    print("🎉 STEP 13P PASSED")
    print("✓ Both ONNX resource models load correctly")
    print("✓ Ready to retry full augmentation")
else:
    print("❌ STEP 13P FAILED")

print("=" * 60)

In [ ]:
import sys
import os
import subprocess

print("=" * 60)
print("STEP 13Q — FULL AUGMENTATION RETRY")
print("=" * 60)
print()
print("Starting full augmentation...")
print("Please let it finish completely.")
print()

env = os.environ.copy()
env["PYTHONPATH"] = "/content/openwakeword:/content"

cmd = [
    sys.executable,
    "-u",
    "/content/openwakeword/openwakeword/train.py",
    "--training_config",
    "/content/hey_lisa_model.yaml",
    "--augment_clips"
]

process = subprocess.run(
    cmd,
    env=env,
    text=True
)

print()
print("=" * 60)

if process.returncode == 0:
    print("🎉 STEP 13Q PASSED")
    print("=" * 60)
    print()
    print("✓ Full augmentation completed successfully")
    print("✓ Training features should now be ready")
else:
    print("❌ STEP 13Q FAILED")
    print("=" * 60)
    print()
    print("Return code:", process.returncode)
    print()
    print("Send me the complete output.")

In [ ]:
from pathlib import Path

print("=" * 60)
print("STEP 13R — VERIFY TRAINING FEATURES")
print("=" * 60)
print()

model_dir = Path("/content/my_custom_model/hey_lisa")

feature_files = [
    "positive_features_train.npy",
    "positive_features_test.npy",
    "negative_features_train.npy",
    "negative_features_test.npy"
]

all_ok = True

for filename in feature_files:
    path = model_dir / filename

    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)

        print(f"✓ {filename}")
        print(f"  Size: {size_mb:.2f} MB")
    else:
        print(f"❌ {filename} NOT FOUND")
        all_ok = False

    print()

print("=" * 60)

if all_ok:
    print("🎉 STEP 13R PASSED")
    print("✓ All four training feature files exist")
    print("✓ Ready to train the Hey Lisa model")
else:
    print("❌ STEP 13R FAILED")
    print("Some feature files are missing.")

print("=" * 60)

In [ ]:
import sys
import os
import subprocess
import time
import re

print("=" * 60)
print("STEP 14 — TRAIN HEY LISA MODEL")
print("=" * 60)
print()

env = os.environ.copy()
env["PYTHONPATH"] = "/content/openwakeword:/content"

cmd = [
    sys.executable,
    "-u",
    "/content/openwakeword/openwakeword/train.py",
    "--training_config",
    "/content/hey_lisa_model.yaml",
    "--train_model"
]

TOTAL_STEPS = 10000
start_time = time.time()
last_step = 0

print("Training started...")
print("Target steps:", TOTAL_STEPS)
print()

process = subprocess.Popen(
    cmd,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end="")

    # Try to detect training step numbers from output
    matches = re.findall(
        r"(?:step|steps)[^\d]{0,10}(\d+)",
        line,
        re.IGNORECASE
    )

    if matches:
        try:
            step = int(matches[-1])

            if 0 <= step <= TOTAL_STEPS and step >= last_step:
                last_step = step

                elapsed = time.time() - start_time

                if step > 0:
                    steps_per_sec = step / elapsed
                    remaining_steps = TOTAL_STEPS - step
                    remaining_sec = remaining_steps / steps_per_sec

                    percent = (step / TOTAL_STEPS) * 100

                    print()
                    print(
                        f"📊 Progress: {step:,}/{TOTAL_STEPS:,} "
                        f"({percent:.1f}%)"
                    )
                    print(
                        f"⏱️ Elapsed: {elapsed/60:.1f} min"
                    )
                    print(
                        f"⚡ Speed: {steps_per_sec:.2f} steps/sec"
                    )
                    print(
                        f"🕐 Estimated remaining: "
                        f"{remaining_sec/60:.1f} min"
                    )
                    print()

        except:
            pass

return_code = process.wait()

elapsed_total = time.time() - start_time

print()
print("=" * 60)

if return_code == 0:
    print("🎉 STEP 14 PASSED")
    print("=" * 60)
    print()
    print("✓ Hey Lisa model training completed")
    print(f"⏱️ Actual training time: {elapsed_total/60:.2f} minutes")
    print()
    print("✓ Ready for model export")
else:
    print("❌ STEP 14 FAILED")
    print("=" * 60)
    print()
    print("Return code:", return_code)
    print(f"⏱️ Time before failure: {elapsed_total/60:.2f} minutes")
    print()
    print("Send me the complete output.")

In [ ]:
from pathlib import Path

print("=" * 60)
print("STEP 15 — CHECK TRAINED ONNX MODEL")
print("=" * 60)
print()

model_path = Path(
    "/content/my_custom_model/hey_lisa.onnx"
)

if model_path.exists():
    size_mb = model_path.stat().st_size / (1024 * 1024)

    print("🎉 HEY LISA ONNX MODEL FOUND!")
    print()
    print("File:")
    print(model_path)
    print()
    print(f"Size: {size_mb:.2f} MB")
    print()
    print("✓ Model training completed")
    print("✓ ONNX model was created")
    print("✓ TFLite conversion failure can be ignored")
else:
    print("❌ hey_lisa.onnx was not found here:")
    print(model_path)
    print()
    print("Let's locate the ONNX file...")

    matches = list(Path("/content").rglob("hey_lisa.onnx"))

    if matches:
        print()
        print("Found:")
        for p in matches:
            print(" ", p)
    else:
        print("No hey_lisa.onnx found.")

print()
print("=" * 60)

In [ ]:
import onnxruntime as ort
from pathlib import Path

print("=" * 60)
print("STEP 16 — VERIFY HEY LISA MODEL")
print("=" * 60)
print()

model_path = Path("/content/my_custom_model/hey_lisa.onnx")

print("Model:", model_path)
print("Size:", f"{model_path.stat().st_size / (1024 * 1024):.2f} MB")
print()

try:
    session = ort.InferenceSession(
        str(model_path),
        providers=["CPUExecutionProvider"]
    )

    print("🎉 MODEL LOADED SUCCESSFULLY")
    print()
    print("Input:")
    for inp in session.get_inputs():
        print(" ", inp.name, inp.shape, inp.type)

    print()
    print("Output:")
    for out in session.get_outputs():
        print(" ", out.name, out.shape, out.type)

    print()
    print("=" * 60)
    print("🎉 STEP 16 PASSED")
    print("✓ hey_lisa.onnx is a valid ONNX model")
    print("✓ ONNX Runtime can load it")
    print("=" * 60)

except Exception as e:
    print()
    print("❌ STEP 16 FAILED")
    print(type(e).__name__ + ":", e)

In [ ]:
from google.colab import files
from pathlib import Path

model_path = Path("/content/my_custom_model/hey_lisa.onnx")

print("=" * 60)
print("STEP 17 — DOWNLOAD HEY LISA MODEL")
print("=" * 60)
print()
print("Model:", model_path)
print("Size:", f"{model_path.stat().st_size / (1024 * 1024):.2f} MB")
print()
print("Starting download...")
print()

files.download(str(model_path))